In [1]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import sys
from pyspark.sql import SparkSession
print("done")

done


In [2]:
spark = SparkSession.builder \
 .master("local") \
 .appName("Project") \
 .getOrCreate()

In [3]:
pwd

'/home/jovyan'

In [5]:
#pollution dataset
pollution_df = spark.read.csv("global air pollution dataset.csv", header=True, inferSchema=True)
pollution_df.show(vertical=True)

-RECORD 0----------------------------------
 Country            | Russian Federation   
 City               | Praskoveya           
 AQI Value          | 51                   
 AQI Category       | Moderate             
 CO AQI Value       | 1                    
 CO AQI Category    | Good                 
 Ozone AQI Value    | 36                   
 Ozone AQI Category | Good                 
 NO2 AQI Value      | 0                    
 NO2 AQI Category   | Good                 
 PM2.5 AQI Value    | 51                   
 PM2.5 AQI Category | Moderate             
-RECORD 1----------------------------------
 Country            | Brazil               
 City               | Presidente Dutra     
 AQI Value          | 41                   
 AQI Category       | Good                 
 CO AQI Value       | 1                    
 CO AQI Category    | Good                 
 Ozone AQI Value    | 5                    
 Ozone AQI Category | Good                 
 NO2 AQI Value      | 1         

In [7]:
pollution_df.printSchema()

root
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- AQI Value: integer (nullable = true)
 |-- AQI Category: string (nullable = true)
 |-- CO AQI Value: integer (nullable = true)
 |-- CO AQI Category: string (nullable = true)
 |-- Ozone AQI Value: integer (nullable = true)
 |-- Ozone AQI Category: string (nullable = true)
 |-- NO2 AQI Value: integer (nullable = true)
 |-- NO2 AQI Category: string (nullable = true)
 |-- PM2.5 AQI Value: integer (nullable = true)
 |-- PM2.5 AQI Category: string (nullable = true)



In [11]:
#GDP per capita data to combine
GDP_df = spark.read.csv("API_NY.GDP.PCAP.CD_DS2_en_csv_v2_245.csv", header=True, inferSchema=True)
GDP_2022_df = GDP_df.select("Country Name", "Country Code", GDP_df["2022"])
GDP_2022_df.show(df.count())

+--------------------+------------+----------------+
|        Country Name|Country Code|            2022|
+--------------------+------------+----------------+
|               Aruba|         ABW|30975.9989121779|
|Africa Eastern an...|         AFE|1679.32762166466|
|         Afghanistan|         AFG|357.261152798144|
|Africa Western an...|         AFW|2138.47315259913|
|              Angola|         AGO| 3682.1131513698|
|             Albania|         ALB|7756.96188744253|
|             Andorra|         AND|42414.0479861228|
|          Arab World|         ARB|7950.35581975672|
|United Arab Emirates|         ARE|50759.7589231191|
|           Argentina|         ARG|13962.1894087223|
|             Armenia|         ARM|6571.97445545099|
|      American Samoa|         ASM|18017.4589383973|
| Antigua and Barbuda|         ATG|20105.1989085164|
|           Australia|         AUS|65169.5191118657|
|             Austria|         AUT|52336.7725223758|
|          Azerbaijan|         AZE|  7770.5942

In [12]:
from pyspark.sql.functions import col

missing_2022_df = GDP_df.filter(col("2022").isNull()) \
                       .select("Country Name", "Country Code")

missing_2022_df.show()

+--------------------+------------+
|        Country Name|Country Code|
+--------------------+------------+
|                Cuba|         CUB|
|             Eritrea|         ERI|
|           Gibraltar|         GIB|
|      Not classified|         INX|
|St. Martin (Frenc...|         MAF|
|Korea, Dem. Peopl...|         PRK|
|         South Sudan|         SSD|
|British Virgin Is...|         VGB|
|         Yemen, Rep.|         YEM|
+--------------------+------------+



In [16]:
# Identify countries that don't have a matching name in both datasets
non_matching_countries = pollution_df.select(col("Country").alias("country_name")).distinct().join(GDP_2022_df.select(col("Country Name").alias("country_name")).distinct(),on="country_name",how="left_anti")
non_matching_countries.show(n=non_matching_countries.count(), truncate=False)


+----------------------------------------------------+
|country_name                                        |
+----------------------------------------------------+
|Côte d'Ivoire                                       |
|Yemen                                               |
|State of Palestine                                  |
|Republic of Korea                                   |
|Turkey                                              |
|Kingdom of Eswatini                                 |
|Republic of North Macedonia                         |
|Slovakia                                            |
|Somalia                                             |
|United Republic of Tanzania                         |
|Bolivia (Plurinational State of)                    |
|Venezuela (Bolivarian Republic of)                  |
|Congo                                               |
|Democratic Republic of the Congo                    |
|Saint Kitts and Nevis                               |
|United St